# 14.01 · Vision-language models

**Chapter:** Multimodal AI · **Milestone:** M7

## Goals
- [ ] Explain how a VLM works: vision encoder → projector → LLM, and what an "image token" costs
- [ ] Send images through the same OpenAI-compatible API (API model and local model)
- [ ] Build a small **document-extraction eval** with exact labels and compare models on accuracy and latency
- [ ] Chart understanding; know the open model landscape (Qwen-VL, Gemma, SmolVLM, InternVL, Llama 4, Mistral) vs closed APIs

## Resources
- [HF – Vision Language Models explained](https://huggingface.co/blog/vlms) · [VLMs 2025 update](https://huggingface.co/blog/vlms-2025) — free
- [Gemini – Image understanding](https://ai.google.dev/gemini-api/docs/image-understanding) — free tier
- [SmolVLM paper](https://arxiv.org/abs/2504.05299) · [Qwen2.5-VL report](https://arxiv.org/abs/2502.13923) — free
- [Ollama vision models](https://ollama.com/search?c=vision) · [mlx-vlm](https://github.com/Blaizzy/mlx-vlm) — free

**How it works.** A vision encoder (a ViT such as SigLIP) cuts the image into patches (e.g. 14×14 px) and embeds each one. A small projector maps
those embeddings into the LLM's token space, and the LLM reads them like (a lot of) extra tokens before your text. Higher resolution → more
patches → more tokens → more cost and latency, but better OCR. Models tile big images, or compress patches (pixel shuffle) to save tokens.

In [ ]:
import base64
import io
import json
import random
import time
import unicodedata
from datetime import date, timedelta

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont
from pydantic import BaseModel

from matplotlib import font_manager

from aieng import DATA
from aieng.llm import available, client

## 1. A labelled dataset for free: synthetic receipts
Same trick as chapter 12: generate the documents, so the labels are exact. Variety on purpose: date formats, currencies, item counts,
and some degraded images (blur, rotation, low resolution), like phone photos.

In [ ]:
VENDORS = ["SUPERMERCADO SOL", "Farmacia Universal", "Café Andino", "TechStore SAC", "Panadería La Espiga", "Grifo Norte", "Librería Central"]
ITEMS = ["Leche 1L", "Pan integral", "Palta", "Cafe molido", "Arroz 5kg", "Aceite", "Huevos x12", "Cuaderno A4", "Cable USB-C", "Galletas"]


# PIL's default font has no "í" or "€": they'd render as boxes and the labels would be unreadable. Always look at your synthetic data.
FONT = ImageFont.truetype(font_manager.findfont("DejaVu Sans Mono"), 18)


def make_receipt(rng):
    d = date(2025, 1, 1) + timedelta(days=rng.randint(0, 600))
    items = [(rng.choice(ITEMS), rng.randint(1, 4), round(rng.uniform(1.5, 60), 2)) for _ in range(rng.randint(2, 7))]
    total = round(sum(q * p for _, q, p in items), 2)
    cur = rng.choice(["S/", "$", "€"])
    date_txt = rng.choice([d.isoformat(), d.strftime("%d/%m/%Y"), d.strftime("%d %b %Y")])
    lines = [rng.choice(VENDORS), f"RUC {rng.randint(10**10, 10**11 - 1)}", date_txt + f"  {rng.randint(7, 22):02d}:{rng.randint(0, 59):02d}", ""]
    lines += [f"{n:<16} x{q}  {q * p:8.2f}" for n, q, p in items] + ["", f"TOTAL   {cur} {total:.2f}"]
    img = Image.new("RGB", (460, 60 + 34 * len(lines)), "white")
    draw = ImageDraw.Draw(img)
    for i, line in enumerate(lines):
        draw.text((24, 24 + 34 * i), line, fill="black", font=FONT)
    hard = rng.random() < 0.35
    if hard:  # simulate a bad phone photo
        img = img.rotate(rng.uniform(-6, 6), expand=True, fillcolor="white").filter(ImageFilter.GaussianBlur(1.2))
        img = img.resize((img.width * 2 // 3, img.height * 2 // 3))
    label = {"vendor": lines[0], "date": d.isoformat(), "total": total, "currency": {"S/": "PEN", "$": "USD", "€": "EUR"}[cur]}
    return img, label, hard


rng = random.Random(0)
docs = [make_receipt(rng) for _ in range(20)]
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
for ax, (img, label, hard) in zip(axes, docs[:3]):
    ax.imshow(img); ax.axis("off"); ax.set_title(f"{'hard' if hard else 'clean'}: {label['total']} {label['currency']}", fontsize=9)
plt.show()
print(sum(h for *_, h in docs), "of", len(docs), "are degraded")

## 2. Images through the OpenAI-compatible API
Images go in the message `content` as a list of parts; base64 data URLs work everywhere (local and API). Same code for Gemini and Ollama.

Local model: `ollama pull qwen3-vl:2b-instruct` (~1.9 GB). Use the **instruct** tag: the default `qwen3-vl:2b` is the *thinking* variant,
which ignores `reasoning_effort` over the OpenAI API and spends hundreds of tokens reasoning before answering (often hitting `max_tokens`).

In [ ]:
def data_url(img):
    buf = io.BytesIO(); img.save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


class Receipt(BaseModel):
    vendor: str
    date: str  # YYYY-MM-DD
    total: float
    currency: str  # PEN | USD | EUR


PROMPT = ('Extract the receipt as JSON only: {"vendor": str, "date": "YYYY-MM-DD", "total": number, "currency": "PEN" | "USD" | "EUR"}. '
          "S/ means PEN.")


def ask_image(provider, model, img, prompt=PROMPT, max_tokens=300):
    c, default = client(provider)
    extra = {"reasoning_effort": "none"} if c.base_url.host == "localhost" else {}
    r = c.chat.completions.create(model=model or default, temperature=0, max_tokens=max_tokens, **extra, messages=[
        {"role": "user", "content": [{"type": "text", "text": prompt}, {"type": "image_url", "image_url": {"url": data_url(img)}}]}])
    return r.choices[0].message.content, r.usage


img, label, _ = docs[0]
for provider, model in [("ollama", "qwen3-vl:2b-instruct"), ("gemini", None)]:
    if provider in available():
        out, usage = ask_image(provider, model, img)
        print(f"{provider:7} {usage.prompt_tokens:5} prompt tokens (incl. image) -> {out.strip()[:150]}")
print("label  ", json.dumps(label))

Note the prompt tokens: one small receipt image costs hundreds of tokens. A full-page scan at high resolution can cost thousands —
resize images to the smallest resolution that keeps your accuracy (exercise 2).

## 3. The eval: field-level accuracy
Score each field separately: it tells you *what* fails (dates? totals on blurry images?), which an overall pass/fail hides.

In [ ]:
def parse(text):
    try:
        s = text[text.index("{"): text.rindex("}") + 1]
        return Receipt.model_validate_json(s)
    except Exception:
        return None


def norm(s):  # case- and accent-insensitive: "Libreria" == "Librería"
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().strip().lower()


def field_scores(pred, label):
    if pred is None:
        return {"valid": 0, "vendor": 0, "date": 0, "total": 0, "currency": 0}
    return {"valid": 1, "vendor": int(norm(pred.vendor) == norm(label["vendor"])), "date": int(pred.date == label["date"]),
            "total": int(abs(pred.total - label["total"]) < 0.01), "currency": int(pred.currency.upper() == label["currency"])}


def evaluate(name, predict, pause=0.0):
    rows, secs = [], []
    for img, label, hard in docs:
        t = time.perf_counter()
        try:
            out = predict(img)
        except Exception as e:  # rate limits etc.
            out = ""; print(name, type(e).__name__)
        secs.append(time.perf_counter() - t)
        rows.append({**field_scores(parse(out), label), "hard": hard})
        time.sleep(pause)
    agg = {k: np.mean([r[k] for r in rows]) for k in ("valid", "vendor", "date", "total", "currency")}
    agg["all (clean)"] = np.mean([all(r[k] for k in ("vendor", "date", "total", "currency")) for r in rows if not r["hard"]])
    agg["all (hard)"] = np.mean([all(r[k] for k in ("vendor", "date", "total", "currency")) for r in rows if r["hard"]])
    print(f"{name:22}", " ".join(f"{k} {v:4.0%}" for k, v in agg.items()), f"| {np.median(secs):.1f}s/doc")
    return agg


results = {"qwen3-vl:2b-instruct (local)": evaluate("qwen3-vl:2b-instruct (local)", lambda img: ask_image("ollama", "qwen3-vl:2b-instruct", img)[0])}
if "gemini" in available():
    results["gemini (API)"] = evaluate("gemini (API)", lambda img: ask_image("gemini", None, img)[0], pause=4)  # free tier: ~15 req/min

Field-level scores make the failure obvious: the 2B model *reads* the receipt fine (vendor, total) but copies the date **as printed**
(`07-SEP-2025`, `01/01/2026`) instead of converting it to ISO. That's not a vision problem — it's instruction following. Cheap fix: ask for the
date as printed and normalise it in code (`dateutil.parser.parse`, with `dayfirst=True` for Peru). Put deterministic work in code, not in the model.

## 4. A tiny local VLM with transformers: SmolVLM-256M
256M parameters, runs anywhere. Small VLMs are great at captions and simple layouts and much weaker at dense OCR; the eval shows exactly how much.

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

device = "mps" if torch.backends.mps.is_available() else "cpu"
SMOL = "HuggingFaceTB/SmolVLM-256M-Instruct"
proc = AutoProcessor.from_pretrained(SMOL)
smol = AutoModelForImageTextToText.from_pretrained(SMOL, dtype=torch.bfloat16).to(device).eval()


def smol_generate(img, prompt=PROMPT, max_new_tokens=120):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    inputs = proc(text=proc.apply_chat_template(msgs, add_generation_prompt=True), images=[img], return_tensors="pt").to(device)
    with torch.no_grad():
        out = smol.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return proc.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]


print("caption:", smol_generate(docs[0][0], "Describe this image in one sentence."))
results["SmolVLM-256M (local)"] = evaluate("SmolVLM-256M (local)", smol_generate)

SmolVLM can caption, but at 256M it doesn't follow the JSON instruction at all: below ~1B parameters, expect to fine-tune for structured tasks.

## 5. Chart understanding
Charts mix OCR (axis labels, numbers) with visual reasoning (which bar is tallest?). Again: generate the chart, so we know the answer.

In [ ]:
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
r = random.Random(1)
values = [r.randint(20, 90) for _ in months]
fig, ax = plt.subplots(figsize=(5, 3)); ax.bar(months, values, color="#4c78a8"); ax.set_title("Monthly sales (units)")
for i, v in enumerate(values):
    ax.text(i, v + 1, str(v), ha="center", fontsize=9)
buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=100, bbox_inches="tight"); plt.close(fig)
chart = Image.open(buf).convert("RGB")
truth = (months[int(np.argmax(values))], max(values))
q = "Which month has the highest sales and what is its value? Answer as 'Month: value' only."
print("truth:", truth)
for provider, model in [("ollama", "qwen3-vl:2b-instruct"), ("gemini", None)]:
    if provider in available():
        print(f"{provider:7}", ask_image(provider, model, chart, q, max_tokens=50)[0].strip())
print("smolvlm", smol_generate(chart, q, 30).strip())

## 6. Choosing a VLM
| | Examples | Strengths | Watch out |
|---|---|---|---|
| Closed APIs | Gemini, GPT-5, Claude | best OCR/reasoning, long PDFs natively (Gemini), no infra | per-image token cost, data leaves your network |
| Open, mid-size | Qwen2.5/3-VL 7–32B, Gemma 3/4, InternVL3, Mistral Small, Llama 4 | strong OCR, self-hostable, fine-tunable | GPU needed for speed |
| Open, small | Qwen3-VL 2B/4B, SmolVLM, Moondream, Florence-2 | runs on a laptop/phone, cheap | weaker on dense or degraded documents |
| OCR pipelines | Docling (08.02), PaddleOCR, Tesseract, Mistral OCR (API) | fast, cheap, layout-aware | no reasoning; brittle on handwriting/photos |

Common production pattern: an OCR/layout pipeline for clean digital documents, a VLM for photos, handwriting, charts, and anything the pipeline
fails to parse — decided by an eval like the one above.

## Exercises
1. Compare an API VLM and a local VLM on 20 **real** document screenshots or phone photos of your own (label them by hand: it's 20 minutes).
2. Downscale the receipts to 50%, 33% and 25% before sending. Plot accuracy and prompt tokens vs resolution for Gemini and qwen3-vl.
3. Implement the date fix above (ask for the date as printed + parse in code) and re-run the qwen3-vl eval. How close does it get to Gemini?
4. Add a "hard mode" with handwriting-like fonts, coffee stains (random brown circles) and perspective warp. Which field breaks first?
5. Pull a bigger local model (`ollama pull qwen3-vl:4b-instruct` or `gemma3:4b`) and add it to the table. Is the accuracy gain worth the latency?

In [ ]:
# your code here